# Parametric PINN (50 impacts) demo

This notebook runs `parametric_pinn_50_impacts.py` from the `parametric PINN/` folder.
It trains one network on `(t, phi1, phi2) -> x(t;phi1,phi2)` and predicts unseen parameters.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import matplotlib.pyplot as plt

module_path = Path('parametric PINN/parametric_pinn_50_impacts.py')
spec = importlib.util.spec_from_file_location('parametric_pinn_50_impacts', module_path)
mod = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mod
spec.loader.exec_module(mod)

TrainingConfig = mod.TrainingConfig
ParametricImpactPINN = mod.ParametricImpactPINN

In [ ]:
# Training setup (adjust epochs for quick trial vs production)
cfg = TrainingConfig(
    phi1_range=(1.0, 2.0),
    phi2_range=(10.0, 20.0),
    n_param_samples=20,
    n_collocation_t=2000,
    t_final=30.0,
    adam_epochs=2000,   # increase (e.g., 4000+) for stronger convergence
    log_every=200,
)

pinn = ParametricImpactPINN(cfg)
pinn.train()

In [ ]:
# Unseen in-range parameter query
phi1_test, phi2_test = 1.37, 16.25
t_eval = np.linspace(0.0, cfg.t_final, 12000)

x_pred = pinn.predict_response(t_eval, phi1_test, phi2_test)
impact_times = pinn.extract_impact_times(t_eval, phi1_test, phi2_test, n_impacts=50)

print('Predicted points:', x_pred.shape[0])
print('Detected impacts:', impact_times.shape[0])
print('First 10 impact times:', impact_times[:10])

In [ ]:
# Plot response and detected impacts
plt.figure(figsize=(12, 4))
plt.plot(t_eval, x_pred, lw=1.0, label='x(t) prediction')
if impact_times.size > 0:
    y_imp = np.interp(impact_times, t_eval, x_pred)
    plt.scatter(impact_times, y_imp, s=15, c='r', label='detected impacts')
plt.xlabel('t')
plt.ylabel('x')
plt.title(f'Parametric PINN prediction at phi1={phi1_test}, phi2={phi2_test}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()